In [24]:
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely as sp
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon
import random
import networkx as nx
import osmnx as ox
import seaborn as sns
import requests, time
import folium
import serpapi
import datetime
from datetime import timedelta
import httpx

### Setup and Helpers

In [20]:
keys=dict()
with open('.env','r') as f:
    for line in f.readlines():
        parsed = line.split("=")
        keys[parsed[0].strip()] = parsed[1].strip()

In [3]:
serpapi_cli = serpapi.Client(api_key=keys['SERPAPI_KEY'])

In [4]:
gdf = gpd.read_file("data/ne_10m_admin_0_countries.shp")
gdf_airports = gpd.read_file("data/ne_10m_airports.shp")
gdf = gdf.set_index('SOVEREIGNT')
gdf = gdf.set_geometry("geometry")

In [5]:
def nearest_place(lat, lon):
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json",
        "zoom": 10,  # 10 = city/town level, 8 = county, 6 = state
    }
    headers = {"User-Agent": "your-app-name"}  # required by Nominatim ToS
    r = requests.get(url, params=params, headers=headers)
    data = r.json()
    return data["display_name"]

In [35]:
def get_nearby_hotels(lat: float, lon: float, radius_m: int = 5000, limit: int = 50) -> list[dict]:
    url = "https://api.tomtom.com/search/2/nearbySearch/.json"
    params = {
        "key": keys['TOMTOM_API_KEY'],
        "lat": lat,
        "lon": lon,
        "radius": radius_m,
        "categorySet": 7314,
        "limit": limit,
    }
    response = httpx.get(url, params=params)
    response.raise_for_status()

    return [
        {
            "name": r["poi"]["name"],
            "phone": r["poi"].get("phone"),
            "website": r["poi"].get("url"),
            "categories": [c["name"] for c in r["poi"].get("classifications", [{}])[0].get("names", [])],
            
            "address": r["address"]["freeformAddress"],
            "street": r["address"].get("streetName"),
            "house_number": r["address"].get("streetNumber"),
            "city": r["address"].get("municipality"),
            "country": r["address"].get("country"),
            "country_code": r["address"].get("countryCode"),
            "postal_code": r["address"].get("postalCode"),
        
            "lat": r["position"]["lat"],
            "lon": r["position"]["lon"],
            "distance_m": r["dist"]
        }
        for r in response.json().get("results", [])
    ]

### Random Walk

In [43]:
G = ox.graph.graph_from_place("Piedmont, California, USA", network_type="drive")

In [88]:
node = random.choice(list(G.nodes))

In [95]:
neighbors = list(G.neighbors(node))
N = len(neighbors)

edges = []
for neighbor in neighbors:
    for key,val in G[node][neighbor].items():
        val['src']=node
        val['dst']=neighbor
        edges.append(val)
edges

[{'osmid': 6400807,
  'highway': 'residential',
  'name': 'Oak Road',
  'oneway': False,
  'reversed': False,
  'length': np.float64(12.510057371376881),
  'src': 10677487586,
  'dst': 53149343},
 {'osmid': 6400807,
  'highway': 'residential',
  'name': 'Oak Road',
  'oneway': False,
  'reversed': True,
  'length': np.float64(107.03307772838524),
  'geometry': <LINESTRING (-122.232 37.816, -122.232 37.817, -122.232 37.817, -122.232 37....>,
  'src': 10677487586,
  'dst': 260707482}]

In [105]:
def random_walk(G, node, target_length_m, length=0):
    if(length >= target_length_m):
        return [node]
        
    neighbors = list(G.neighbors(node))
    if(len(neighbors) == 0):
        return [node]

    edges = []
    for neighbor in neighbors:
        for key,val in G[node][neighbor].items():
            val['src']=node
            val['dst']=neighbor
            edges.append(val)
        
    p = np.array([edge['length'] for edge in edges])
    p = p/p.sum()
    idx = np.random.choice(len(edges), p=p)
    
    neighbor = edges[idx]['dst']
    return [node] + random_walk(G, neighbor, target_length_m, length + edges[idx]['length'])

In [109]:
path = random_walk(G, node, 10000)

In [110]:
path_gdf = ox.routing.route_to_gdf(G, path)

In [111]:
path_gdf.explore()

### Journey 

In [39]:
## Parameters
start_date = datetime.date(2026, 6, 28)
end_date = start_date + timedelta(days=7)
daily_walk_m = 20_000

In [7]:
sov_idx = np.random.randint(gdf.shape[0])
sov_geom = gdf.iloc[sov_idx].geometry
sov_airports = [p for p in gdf_airports.geometry.intersection(sov_geom) if not p.is_empty]

In [28]:
airport_idx = np.random.randint(len(sov_airports))
airport = sov_airports[airport_idx]

In [36]:
lon, lat = airport.coords[0]
hotels = get_nearby_hotels(lat, lon)

In [38]:
hotel_idx = np.random.randint(len(hotels))
hotel = hotels[hotel_idx]

In [41]:
lat = hotel['lat']
lon = hotel['lon']
G = ox.graph_from_point((lat,lon), dist=daily_walk_m, network_type="walk", custom_filter='["highway"!~"motorway|trunk"]')

KeyboardInterrupt: 

In [ ]:
ox.plot_graph(G)